<a href="https://colab.research.google.com/github/marius-ne/CIE_ProjectB_Group13/blob/colab/Program_B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CIE 2025/26 RWTH, PROJECT B, GROUP 13
Junchao Yu, Marius Neuhalfen

ToDo:
- Find defective nodes by comparing perfect structure and imperfect structures for all scenarios
- Group defective nodes into regions (arc sections or track sections)
- Predict whether structure is perfect or imperfect
- Predict where the imperfection lies

There are 25.XXX for deformation and 24.XXX for stress

# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from _import import *

In [ ]:
I_AM_ON_COLAB = False

## Data getting (Colab only)


In [ ]:
if I_AM_ON_COLAB:
    import google.colab
    google.colab.drive.mount("/content/drive")

Get ancillary data from Github

In [ ]:
if I_AM_ON_COLAB:
    !git clone https://github.com/marius-ne/CIE_ProjectB_Group13.git

## Load data (Colab only)

In [ ]:
if I_AM_ON_COLAB:
    os.chdir("CIE_ProjectB_Group13")
    os.getcwd()

**IMPORTANT:** You need to add a shortcut of the "programB" folder on GoogleDrive to your own "MyDrive" for this to work

In [ ]:
if I_AM_ON_COLAB:
    !ln -s /content/drive/MyDrive/programB data

# Go to data folder (Colab only)

In [ ]:
if I_AM_ON_COLAB:
    target_folder = f"{DATA_FOLDER_PATH}"
    current_folder = os.getcwd()

    if Path(current_folder).name != target_folder:
        os.chdir(Path(current_folder) / Path(target_folder))
    print(os.getcwd())


# Read data

In [ ]:
# df = get_data_variable_aggregated((0, 0, 0, 0), drop_invalid_nodes=True)

In [ ]:
# df = get_data_variable_and_region_aggregated((3,0,0),drop_invalid_nodes=True)

In [ ]:
# df = get_data_variable_and_region_and_season_aggregated((3,0),drop_invalid_nodes=True)

In [ ]:
# df = get_data_variable_and_region_and_season_and_load_aggregated((3,), drop_invalid_nodes=True)

In [ ]:
df_raw = get_data_all_aggregated(
    filter_invalid_nodes=True, 
    drop_invalid_nodes=False,
    filter_negative_total_deformation=False
)

In [ ]:
# Show log histogram
# df["DirectionalDeformation_Z_axis"].hist(bins=100, figsize=(10,5), log=True)
# df.loc[df["DirectionalDeformation_Z_axis"] < -2.5, "DirectionalDeformation_Z_axis"] = 0

In [ ]:
df = df_raw.copy()

Select preprocessing

In [ ]:
CLASSIFIER_TO_USE = "XGBoost"  # "XGBoost", "AE", "NN", "RandomForest", "DecisionTree" 

USE_ONLY_Z_DEFORMATION = True
SUBTRACT_MEAN_Z = False
ADD_LOCAL_INFORMATION = True
USE_PEAK_VALUE_PER_TIMESTEPS = False

DELTA_NODES_FROM = "both" # "graphic", "data", or "both"
DROP_NODE_NUMBERS = True
DROP_COORDINATES = False

REMOVE_HEALTHY_SCENARIOS = True
USE_SMOTE = False

if not DROP_NODE_NUMBERS and USE_SMOTE:
    raise ValueError("SMOTE cannot be used when node numbers are kept.")

# Take only deformation if desired

Take only Z deformation

In [ ]:
if USE_ONLY_Z_DEFORMATION:
    new_cols = [
        col for col in df.columns if \
        (col not in VARIABLE_NAMES) or \
        (col == "DirectionalDeformation_Z_axis")
        # (col == "TotalDeformation")
    ]
    df = df[new_cols]

Normalize z per scenario

In [ ]:
if SUBTRACT_MEAN_Z:
    # Calculate the average Z-deflection of the whole bridge for each specific time step
    means = df.groupby(["scenario", "time"])["DirectionalDeformation_Z_axis"].transform("mean")
    # df["DirectionalDeformation_Z_axis"] = df["DirectionalDeformation_Z_axis"] - means
    df["DirectionalDeformation_Z_axis"] = df["DirectionalDeformation_Z_axis"] - means

Take only peak values

In [ ]:
if USE_PEAK_VALUE_PER_TIMESTEPS:
    value_cols = [col for col in VARIABLE_NAMES if col in df.columns]
    agg_dict = {c: "max" for c in value_cols}
    # keep coordinates and health (choose 'first' or 'mean' depending on desired behavior)
    agg_dict.update({"X": "first", "Y": "first", "Z": "first"})
    df = df.groupby(["scenario", "Node Number"]).agg(agg_dict).reset_index()
    df["time"] = 0  # dummy time column

Add local information

In [ ]:
if ADD_LOCAL_INFORMATION:
    # 1. Load the neighbor mapping
    with open('node_neighbors_50.pkl', 'rb') as f:
        neighbor_map = pickle.load(f)

    print("Calculating local neighbor means...")

    # 2. Pivot to Wide Format (Rows=Snapshots, Cols=Nodes)
    # This aligns all nodes in time so we can do vectorized math
    # Shape: (n_scenarios * n_timesteps, n_nodes)
    df_wide = df.pivot_table(
        index=['scenario', 'time'], 
        columns='Node Number', 
        values='DirectionalDeformation_Z_axis'
    )

    # 3. Compute the Mean of Neighbors for every node
    # Create a placeholder dataframe with the same shape as df_wide
    local_mean_wide = pd.DataFrame(index=df_wide.index, columns=df_wide.columns)
    local_std_wide = pd.DataFrame(index=df_wide.index, columns=df_wide.columns)

    # Iterate over columns (nodes) only - much faster than iterating rows
    for node_id in df_wide.columns:
        if node_id in neighbor_map:
            neighbors = neighbor_map[node_id]
            
            # Filter for neighbors that actually exist in the current dataframe columns
            # (Safety check in case of subsetting)
            valid_neighbors = [n for n in neighbors if n in df_wide.columns]
            
            if valid_neighbors:
                # Calculate row-wise mean for these specific neighbor columns
                local_mean_wide[node_id] = df_wide[valid_neighbors].mean(axis=1)
                local_std_wide[node_id] = df_wide[valid_neighbors].std(axis=1)

    # 4. Melt back to Long Format to merge with original data
    local_mean_long = local_mean_wide.reset_index().melt(
        id_vars=['scenario', 'time'],
        var_name='Node Number',
        value_name='Z_local_mean'
    )
    local_std_long = local_std_wide.reset_index().melt(
        id_vars=['scenario', 'time'],
        var_name='Node Number',
        value_name='Z_local_std'
    )

    # Ensure data types match for merging
    local_mean_long['Node Number'] = local_mean_long['Node Number'].astype(df['Node Number'].dtype)
    local_std_long['Node Number'] = local_std_long['Node Number'].astype(df['Node Number'].dtype)

    # 5. Merge and Create the Feature
    # Merge the mean back onto the main dataframe
    df = pd.merge(df, local_mean_long, on=['scenario', 'time', 'Node Number'], how='left')
    df = pd.merge(df, local_std_long, on=['scenario', 'time', 'Node Number'], how='left')

    # CALCULATE THE KEY FEATURE:
    # The difference between the node's position and the average of its neighbors.
    # This reveals "kinks" or local failures regardless of total load.
    df['Z_neighbor_residual'] = df['DirectionalDeformation_Z_axis'] - df['Z_local_mean']
    if 'Z_local_mean' not in VARIABLE_NAMES:
        VARIABLE_NAMES.extend(['Z_local_mean', 'Z_local_std', 'Z_neighbor_residual'])

    print("Done. Added columns: 'Z_local_mean', 'Z_local_std', and 'Z_neighbor_residual'")
    df.head()

# Filter delta nodes

## Take from graphics (node-wise health)

In [ ]:
if DELTA_NODES_FROM == "graphic":
    df = add_graphic_delta_nodes(df)

## Take from delta (node-wise health)

In [ ]:
if DELTA_NODES_FROM == "data":
    df = filter_df_to_delta_nodes(
        df, 
        # top_pct=500, 
        top_pct=1, 
        variable_names=["DirectionalDeformation_Z_axis"],
        aggregate_by_time=True
    )[0]

## Merge both

In [ ]:
if DELTA_NODES_FROM == "both":
    # Work on shallow copies to avoid reindexing or row loss
    df_graphic = df.copy(deep=False)
    df_data = df.copy(deep=False)

    # 1) Compute graphic-based delta flags
    df_graphic = add_graphic_delta_nodes(df_graphic)

    # 2) Compute data-based delta flags (returns (df_with_flags, delta_nodes, diffs))
    df_data, _, _ = filter_df_to_delta_nodes(
        df_data,
        top_pct=500,
        variable_names=["DirectionalDeformation_Z_axis"],
        aggregate_by_time=True
    )

    # 3) Build stable row keys on original df to align masks safely
    key_cols = ["Node Number", "scenario", "time"]
    df_key = pd.MultiIndex.from_frame(df[key_cols])

    graphic_key = pd.MultiIndex.from_frame(
        df_graphic.loc[df_graphic["delta_health"] == 0, key_cols]
    )
    data_key = pd.MultiIndex.from_frame(
        df_data.loc[df_data["delta_health"] == 0, key_cols]
    )

    # Intersection (nodes flagged by BOTH methods). Use union() if you prefer either method.
    both_key = graphic_key.intersection(data_key)

    # 4) Create final mask aligned to df rows
    final_mask = pd.Series(df_key.isin(both_key), index=df.index)

    # 5) Write delta_health without touching row order
    df["delta_health"] = np.ones(len(df), dtype=np.int8)
    df.loc[final_mask, "delta_health"] = 0

    # Diagnostics
    print(f"Graphic-only flagged rows: {len(graphic_key)}")
    print(f"Data-only flagged rows:    {len(data_key)}")
    print(f"Intersection rows:         {final_mask.sum()}")

    del df_graphic, df_data

Remove healthy scenarios

In [ ]:
if REMOVE_HEALTHY_SCENARIOS:
    scenarios = df["scenario"].unique().tolist()
    for scenario in scenarios:
        scenario_mask = df["scenario"] == scenario
        region = scenario_number_to_combination(scenario)[-1]
        if region == 0:
            # No defective nodes in region 0
            df.drop(index=df[scenario_mask].index, inplace=True)

# Data inspection

Compare stresses

In [ ]:
# df_stress_healthy = read_data_file(0, 0, 0, 0, 4, filter_out_invalid_nodes=True)
# df_stress_unhealthy = read_data_file(0, 0, 0, 6, 4, filter_out_invalid_nodes=True)
# df_stress_healthy

## Obtain delta nodes

Note: there is a large difference between scenarios (3, 1, 1) (179 delta nodes) and (1, 1, 1) (0 delta nodes)!

In [ ]:
# delta_nodes, diffs = get_delta_nodes(top_pct=0.0001)

In [ ]:
# {k:len(v) for k, v in delta_nodes.items()}, len(set(np.concatenate(list(delta_nodes.values()))))

Get superset of delta nodes

In [ ]:
# delta_nodes_superset = set()
# for nodes in delta_nodes.values():
#     delta_nodes_superset.update(nodes)

# delta_nodes_superset = [int(x) for x in np.asarray(list(delta_nodes_superset)).tolist()]
# # with open("delta_nodes_superset_0.0001'_top_pct.pkl", "wb") as f:
#     # pickle.dump(delta_nodes_superset, f)

In [ ]:
# df = get_data_variable_aggregated((0,0,0,0), filter_out_invalid_nodes=True)
# plot_bridge_3d_variable_over_time_df(df, "TotalDeformation")


In [ ]:
# combo = (2, 0, 1, 2, 3)
# var_name = VARIABLE_NAMES[combo[-1]]
# diff = get_variable_difference_between_combinations(
#     (*combo[:3], 0, combo[-1]),
#     combo,
#     top_pct=0.01
# )
# plot_bridge_3d_variable_over_time_df(diff, var_name)

Compare deformations

In [ ]:
# df_diff = get_variable_difference_between_combinations((0, 0, 0, 0, 0), (0, 0, 0, 1, 0), "TotalDeformation")

In [ ]:
# plot_bridge_3d_variable_over_time_df(df_diff, "TotalDeformation")

# Visualize bridge structure

Show deformation only for changing nodes (i.e. )

In [ ]:
# combo = (0, 1, 1, 6, 1)
# var = VARIABLE_NAMES[combo[4]]
# df_agg = get_data_variable_aggregated(combo[:-1])

In [ ]:
# plot_bridge_3d_variable_over_time_df(
#     df_agg, "TotalDeformation", title=combination_to_string(combo)
# )

In [ ]:
# plot_bridge_3d_structure(highlight_nodes=delta_nodes_superset)

# Safety checks

In [ ]:
print(f"Number of healthy nodes: {len(df[df['delta_health']==1])}, number of damaged nodes: {len(df[df['delta_health']!=1])}")

In [ ]:
# Verify that no two rows share the same (Node Number, time, scenario)
keys = ["Node Number", "time", "scenario"]
dupes = df[df.duplicated(subset=keys, keep=False)]
n_dupes = len(dupes)

if n_dupes == 0:
    print("OK: no duplicate (Node Number, time, scenario) combinations found.")
else:
    dup_groups = dupes.groupby(keys).size().reset_index(name="count").sort_values("count", ascending=False)
    print(f"FOUND DUPLICATES: {n_dupes} rows in {len(dup_groups)} duplicated (Node Number, time, scenario) groups.\n")
    print("Top duplicated groups (up to 20):")
    print(dup_groups.head(20).to_string(index=False))
    print("\nExample duplicated rows (up to 20):")
    print(dupes.sort_values(keys).head(20).to_string(index=False))
    raise AssertionError(
        f"Duplicate (Node Number, time, scenario) combinations detected: {len(dup_groups)} groups, {n_dupes} rows."
    )
del n_dupes

In [ ]:
# Show all rows where at least one entry is NaN
nan_rows = df[df.isna().any(axis=1)]
print(f"Found {len(nan_rows)} rows with at least one NaN.\n")
nan_rows

# Also print NaN counts per column for quick overview
print("\nNaN counts per column:")
print(df.isna().sum())
del nan_rows

# Training

Check scenarios

In [ ]:
df["scenario"].unique(), len(df["scenario"].unique())

Choose objective, either scenario-wise health or nodal-wise health

In [ ]:
# y_var = "health"
y_var = "delta_health"

Standardize

In [ ]:
variable_data = df.drop(columns=["Node Number", "time", "scenario", y_var])
x_cols_std = variable_data.columns.tolist()
variable_data_standardized, scaler = standardize(variable_data)

# Add indicators, node number and time back on
for col in ["Node Number", "time", "scenario", y_var]:
    variable_data_standardized[col] = df[col]
del df # save RAM

# columns = ["Node Number", y_var, "time", "X", "Y", "Z"] + VARIABLE_NAMES 
present_variable_names = [var for var in VARIABLE_NAMES if var in variable_data_standardized.columns]
coord_cols = ["X", "Y", "Z"] if not DROP_COORDINATES else []
columns = ["Node Number", "time", "scenario", y_var] + present_variable_names + coord_cols
variable_data_standardized = variable_data_standardized[columns]
variable_data_standardized

## Dimensionality reduction

Select certain nodes

In [ ]:
# # node_subset = set(NODES_MISSING_STRESS) - set(NODES_MISSING_DEFORMATION)

# with open("delta_nodes_new_data_5_per_delta.pkl", "rb") as f:
#     delta_nodes_superset = pickle.load(f)
# node_subset = delta_nodes_superset

# # Create X_raw
# X_filtered = variable_data_standardized[variable_data_standardized['Node Number'].isin(node_subset)]

PCA

In [ ]:
# X_pca, y, pca, scaler = apply_bridge_pca(
#     variable_data_standardized, n_components=50
# )


Drop NaN

In [ ]:
# X_filtered.dropna(inplace=True)

Use all the nodes

In [ ]:
X_filtered = variable_data_standardized

Run this always

In [ ]:

if EVALUATION_FORMAT == "nodal":
    X_wide = reshape_multi_variable_to_wide_nodes(
        X_filtered, y_var_name=y_var, use_coords=not DROP_COORDINATES
    )
    y_raw = X_wide[y_var]
elif EVALUATION_FORMAT == "temporal":
    X_wide = reshape_multi_variable_to_wide(X_filtered)
    
    # Align y to the same row order as X_wide after reshape_multi_variable_to_wide
    # Assume X_wide has columns 'scenario' and 'time' that match health_by_scenario_time_long
    # Align health_by_scenario_time_long to the same row order as X_wide
    # TODO FIX THIS
    health_by_scenario_time_long = health_by_scenario_time_long.set_index(['scenario', 'time']).loc[
        X_wide.set_index(['scenario', 'time']).index
    ].reset_index()

    y_raw = health_by_scenario_time_long[y_var]

# Free RAM
del X_filtered, variable_data_standardized

y_binary = (y_raw > 0).astype(int)

X_wide

## Training

Remove ordinal data again (was used before just to seperate scenarios in the wide format)

In [ ]:
# Split by scenario (group-aware, stratified)
X_train, X_test, y_train, y_test, scenarios_train, scenarios_test = split_by_scenario(
    X_wide, y_raw, scenario_col='scenario', test_size=0.25, random_state=20000, 
)
regions_train = []
for sc_tr in scenarios_train:
    combo = scenario_number_to_combination(sc_tr)
    region = combo[-1]
    regions_train.append(region)
regions_test = []
for sc_tr in scenarios_test:
    combo = scenario_number_to_combination(sc_tr)
    region = combo[-1]
    regions_test.append(region)

# Check that each region appears at least 112/7 times in train/test
from collections import Counter
num_scenarios = len(scenarios_train) + len(scenarios_test)
counter_train = Counter(regions_train)
counter_test = Counter(regions_test)
for region in range(1,7):
    n_train = counter_train.get(region, 0)
    n_test = counter_test.get(region, 0)
    # Show the counters
    print(f"Region {region}: Train={n_train}, Test={n_test}")
    assert n_train >= num_scenarios / 10, f"Region {region} appears only {n_train} times in training set."
    # assert n_test >= num_scenarios / 7, f"Region {region} appears only {n_test} times in test set."


Visualize choices

In [ ]:
from plotting import plot_scenario_tree
test_combos = [scenario_number_to_combination(s) for s in scenarios_test]
train_combos = [scenario_number_to_combination(s) for s in scenarios_train]
plot_scenario_tree(train_combos, test_combos)

In [ ]:
X_train

In [ ]:

# Free RAM
del X_wide, y_raw

train_masks = {s: X_train["scenario"].eq(s) for s in scenarios_train}
test_masks = {s: X_test["scenario"].eq(s) for s in scenarios_test}

X_train_by_scenario = {s: X_train[mask] for s, mask in train_masks.items()}
y_train_by_scenario = {s: y_train[mask] for s, mask in train_masks.items()}
X_test_by_scenario = {s: X_test[mask] for s, mask in test_masks.items()}
y_test_by_scenario = {s: y_test[mask] for s, mask in test_masks.items()}

print(X_train.groupby(y_train).mean())

In [ ]:
node_numbers_train = X_train["Node Number"]
node_numbers_test = X_test["Node Number"]

if DROP_NODE_NUMBERS:
    X_train.drop(columns=[y_var,"scenario","Node Number"], inplace=True)
    X_test.drop(columns=[y_var,"scenario","Node Number"], inplace=True)
else:
    X_train.drop(columns=[y_var,"scenario"], inplace=True)
    X_test.drop(columns=[y_var,"scenario"], inplace=True)
X_train, X_test

## SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE
imbalanced_ratio = float(np.sum(y_train == 1)) / np.sum(y_train == 0)

if USE_SMOTE:
    print("Before SMOTE:", np.bincount(y_train))

    smote = SMOTE(
        sampling_strategy="auto",  # make minority == majority
        k_neighbors=5,             # try 3 if you have very few positives
        random_state=42
    )

    X_train, y_train = smote.fit_resample(X_train, y_train)
    print("After SMOTE:", np.bincount(y_train))

## XGBoost

In [ ]:
import matplotlib.pyplot as plt
from xgboost import XGBClassifier

if CLASSIFIER_TO_USE == "XGBoost":
    model = XGBClassifier(
        n_estimators=125,
        max_depth=10,
        learning_rate=0.02,
        subsample=0.99,
        colsample_bytree=0.99,
        objective="binary:logistic",
        # Adding logloss allows you to see the actual 'Loss' curve
        eval_metric=["auc", "logloss", "rmse"], 
        early_stopping_rounds = 20,
        scale_pos_weight=imbalanced_ratio,
        n_jobs=-1,
        random_state=42,
    )

    model.fit(
        X_train,
        y_train,
        # Include BOTH train and test to get both curves
        eval_set=[(X_train, y_train), (X_test, y_test)],
    )

    # Retrieve performance metrics
    results = model.evals_result()

In [ ]:
if CLASSIFIER_TO_USE == "XGBoost":
    # Plot training & validation AUC and Logloss values
    epochs = len(results['validation_0']['auc'])
    x_axis = range(0, epochs)

    fig, ax = plt.subplots(2, 1, figsize=(6, 8))

    # Plot AUC (Higher is better)
    ax[0].plot(x_axis, results['validation_0']['auc'], label='Train')
    ax[0].plot(x_axis, results['validation_1']['auc'], label='Test')
    ax[0].legend()
    ax[0].set_ylabel('AUC')
    ax[0].set_title('XGBoost AUC Curve')

    # Plot Logloss (Lower is better)
    ax[1].plot(x_axis, results['validation_0']['logloss'], label='Train')
    ax[1].plot(x_axis, results['validation_1']['logloss'], label='Test')
    ax[1].legend()
    ax[1].set_ylabel('Logloss')
    ax[1].set_xlabel('#Estimators')
    ax[1].set_title('XGBoost Logloss Curve')
    best_val_iteration = np.argmin(results['validation_1']['logloss'])
    ax[1].vlines(x=best_val_iteration, ymin=min(results['validation_0']['logloss']), ymax=max(results['validation_0']['logloss']), colors='r', linestyles='dashed', label=f'Best Iteration: {best_val_iteration}')
    ax[1].legend()

    fig.savefig("visualization/xgboost_training_validation_curves.svg", dpi=1000)
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
from xgboost import plot_tree

# Plot the first tree (index 0)
# You can change num_trees to see different trees in the ensemble
fig, ax = plt.subplots(figsize=(30, 30))
plot_tree(model, num_trees=0, ax=ax)
plt.savefig("visualization/xgboost_tree_0.svg", dpi=1000)
plt.show()

In [ ]:
from xgboost import plot_importance
if CLASSIFIER_TO_USE == "XGBoost":
    ax = plot_importance(model, max_num_features=20, importance_type="gain", values_format="{v:.2f}")
    ax.figure.tight_layout()
    ax.title.set_text("XGBoost Feature Importance (Gain)")
    plt.savefig("visualization/xgboost_feature_importance.svg", dpi=300)

## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
if CLASSIFIER_TO_USE == "RandomForest":
    model = RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced",
        verbose=2,
        n_jobs=-1,
        random_state=42,
    )
    model.fit(X_train, y_train)

## Decision Tree

In [ ]:
import itertools
import threading
import time

if CLASSIFIER_TO_USE == "DecisionTree":
    def _spinner(stop_event):
        for c in itertools.cycle("|/-\\"):
            if stop_event.is_set():
                break
            print(f"\rTraining decision tree... {c}", end="", flush=True)
            time.sleep(0.1)

    class DecisionTreeWithProgress(sklearn.tree.DecisionTreeClassifier):
        def fit(self, X, y, **kwargs):
            stop_event = threading.Event()
            t = threading.Thread(target=_spinner, args=(stop_event,), daemon=True)
            t.start()
            try:
                return super().fit(X, y, **kwargs)
            finally:
                stop_event.set()
                t.join()
                print("\rTraining decision tree... done. ", flush=True)

    model = DecisionTreeWithProgress()
    model.fit(X_train, y_train)

## AE

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score

if CLASSIFIER_TO_USE == "AE":
    print("Initializing Autoencoder for Anomaly-based Localization...")
    
    # 1. Prepare Data: Filter ONLY healthy nodes for training
    # Assuming y=1 is Healthy, y=0 is Damaged
    X_train_healthy = X_train[y_train == 1].values.astype('float32')
    X_test_all = X_test.values.astype('float32')
    y_test_all = y_test.values.astype('float32')
    
    input_dim = X_train_healthy.shape[1]

    # 2. Build the Bottleneck Architecture
    def build_ae(dim):
        inp = layers.Input(shape=(dim,))
        
        # Encoder: Compressing the signal
        e = layers.Dense(128)(inp)
        e = layers.LeakyReLU(alpha=0.1)(e)
        e = layers.BatchNormalization()(e)
        
        e = layers.Dense(64)(e)
        e = layers.LeakyReLU(alpha=0.1)(e)
        
        # Latent Space (The bottleneck)
        latent = layers.Dense(16, name="bottleneck")(e) 
        
        # Decoder: Attempting to reconstruct the healthy signal
        d = layers.Dense(64)(latent)
        d = layers.LeakyReLU(alpha=0.1)(d)
        d = layers.BatchNormalization()(d)
        
        d = layers.Dense(128)(d)
        d = layers.LeakyReLU(alpha=0.1)(d)
        
        out = layers.Dense(dim, activation='linear')(d)
        return models.Model(inputs=inp, outputs=out)

    model = build_ae(input_dim)
    model.compile(optimizer=optimizers.Adam(learning_rate=0.001), loss='mse')

    # 3. Training (Learning "Healthy" patterns)
    # Notice: Input and Output are the same (X_train_healthy)
    print(f"Training on {len(X_train_healthy)} healthy samples...")
    history = model.fit(
        X_train_healthy, X_train_healthy,
        epochs=100,
        batch_size=64,
        validation_split=0.1,
        verbose=1,
        callbacks=[callbacks.EarlyStopping(patience=10, restore_best_weights=True)]
    )

    # 4. Define Threshold for Damage (Localization)
    # We calculate the error on the training set. 
    # Any error significantly higher than this indicates an anomaly.
    X_train_pred = model.predict(X_train_healthy)
    train_mse = np.mean(np.square(X_train_healthy - X_train_pred), axis=1)
    
    # We set the threshold at the 99th percentile of healthy error
    THRESHOLD = np.percentile(train_mse, 99)
    print(f"\nCalculated Anomaly Threshold: {THRESHOLD:.6f}")

    # 5. Evaluate on Test Set
    X_test_pred = model.predict(X_test_all)
    test_mse = np.mean(np.square(X_test_all - X_test_pred), axis=1)
    
    # Anomaly scores: Higher error = more likely damaged
    # Logic: if error > THRESHOLD, predict Damaged (0), else Healthy (1)
    y_pred_test = np.where(test_mse > THRESHOLD, 0, 1)

    print("\n--- Autoencoder Detection Results ---")
    print(f"Precision (Class 0): {precision_score(y_test_all, y_pred_test, pos_label=0):.4f}")
    print(f"Recall (Class 0):    {recall_score(y_test_all, y_pred_test, pos_label=0):.4f}")
    print(f"F1 Score (Class 0):  {f1_score(y_test_all, y_pred_test, pos_label=0):.4f}")
    
    # Save the model and threshold for use in the test loop
    model.save("bridge_autoencoder.keras")

## Neural Network

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers, metrics
import numpy as np

# --- 1. Custom Focal Loss ---
# This is the "Magic Sauce" that makes NNs competitive with XGBoost on imbalanced data.
# It down-weights easy examples so the gradient is driven by the rare 'damaged' cases.
class BinaryFocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=0.25, name='binary_focal_loss'):
        super().__init__(name=name)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        # Ensure data types match
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        
        # Clip probabilities to prevent log(0) error
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)

        # Calculate p_t
        p_t = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        alpha_factor = tf.where(tf.equal(y_true, 1), self.alpha, 1 - self.alpha)
        modulating_factor = tf.pow(1.0 - p_t, self.gamma)

        # Formula: -alpha * (1-pt)^gamma * log(pt)
        return -tf.reduce_mean(alpha_factor * modulating_factor * tf.math.log(p_t))

if CLASSIFIER_TO_USE == "NN":
    # 1. Setup
    tf.keras.backend.clear_session()
    
    # Ensure inputs are float32
    X_tr = X_train.values.astype("float32")
    X_te = X_test.values.astype("float32")
    y_tr = y_train.values.astype("float32")
    y_te = y_test.values.astype("float32")
    
    input_dim = X_tr.shape[1]

    # 2. Bias Initialization (Still important!)
    # Helps the model start at the correct "average" probability
    count_0 = np.sum(y_tr == 0) # Damaged
    count_1 = np.sum(y_tr == 1) # Healthy
    # If model predicts 1 (Healthy), bias should favor 1.
    initial_bias = np.log([count_1 / count_0])
    
    print(f"Damaged: {count_0}, Healthy: {count_1}")
    print(f"Initializing output bias to: {initial_bias[0]:.4f}")

    # 3. Optimized Architecture for Tabular Data
    # Smaller is better here. 1024 was likely overfitting noise.
    def create_model(input_dim, output_bias=None):
        if output_bias is not None:
            output_bias = tf.keras.initializers.Constant(output_bias)
            
        inp = layers.Input(shape=(input_dim,))
        
        # Block 1
        x = layers.Dense(256)(inp)
        x = layers.LeakyReLU(alpha=0.1)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.3)(x)

        # Block 2
        x = layers.Dense(128)(x)
        x = layers.LeakyReLU(alpha=0.1)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.2)(x)
        
        # Block 3
        x = layers.Dense(64)(x)
        x = layers.LeakyReLU(alpha=0.1)(x)
        x = layers.BatchNormalization()(x)

        # Output
        out = layers.Dense(1, activation="sigmoid", bias_initializer=output_bias)(x)
        
        return models.Model(inputs=inp, outputs=out)

    model = create_model(input_dim, output_bias=initial_bias)
    
    # 4. Compilation with FOCAL LOSS
    # Note: When using Focal Loss, we often do NOT need class_weights anymore.
    # The 'alpha' parameter in FocalLoss handles the balance.
    # alpha=0.25 generally works well to down-weight the majority class.
    optimizer = optimizers.Adam(learning_rate=0.0005) # Lower LR for stability
    
    model.compile(
        optimizer=optimizer, 
        loss=BinaryFocalLoss(gamma=2.0, alpha=0.25), # Try gamma=2, alpha=0.25
        metrics=[
            metrics.BinaryAccuracy(name='accuracy'),
            metrics.AUC(name='roc_auc'),
            metrics.Precision(name='precision'),
            metrics.Recall(name='recall')
        ]
    )

    # 5. Callbacks
    cb = [
        callbacks.EarlyStopping(
            monitor="val_auc", 
            mode="max", 
            patience=10, 
            restore_best_weights=True,
            verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_auc", 
            mode="max", 
            factor=0.2, 
            patience=8, 
            min_lr=1e-6,
            verbose=1
        )
    ]
    batch_size = 64
    # 6. Training
    # Standard batch size (32-64) often generalizes better than huge batches for tabular data
    print("Starting training with Focal Loss...")
    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_te, y_te),
        epochs=50,
        batch_size=batch_size, 
        callbacks=cb,
        verbose=1
    )
    
    # 7. Evaluate
    print("Final Evaluation:")
    model.evaluate(X_te, y_te)

In [ ]:
if CLASSIFIER_TO_USE in ("NN", "AE"):
    # Plot train/validation loss curves from TensorFlow training
    plt.figure(figsize=(6, 4))
    plt.plot(history.history.get("loss", []), label="Train Loss")
    plt.plot(history.history.get("val_loss", []), label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("TensorFlow Training/Validation Loss")
    plt.legend()
    plt.tight_layout()
    plt.show()

## Isolation Forest

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
import numpy as np

# Change this if you want to use a specific name in your loop
if CLASSIFIER_TO_USE == "IsolationForest": 
    print("Initializing Isolation Forest for Structural Anomaly Detection...")

    # 1. Prepare Data
    # Isolation Forest can be trained on the whole set, but training on 
    # Healthy data only is often more robust for bridge monitoring.
    X_train_healthy = X_train[y_train == 1].values
    X_test_all = X_test.values
    y_test_all = y_test.values

    # 2. Configure the Model
    # 'contamination' is the expected % of damaged nodes. 
    # If you don't know it, 'auto' works, but setting it to your 
    # actual training imbalance ratio (e.g., 0.01) is much better.
    expected_contamination = np.sum(y_train == 0) / len(y_train)
    
    model = IsolationForest(
        n_estimators=200,          # More trees = more stable scores
        max_samples='auto', 
        contamination=expected_contamination, # Use the known training ratio
        random_state=42,
        n_jobs=-1                  # Use all CPU cores
    )

    # 3. Fit the model
    print(f"Fitting Isolation Forest on {len(X_train_healthy)} healthy samples...")
    model.fit(X_train_healthy)

    # 4. Predict
    # Isolation Forest returns: 1 for Inliers (Healthy), -1 for Outliers (Damaged)
    raw_preds = model.predict(X_test_all)
    
    # Convert to your format: 1 = Healthy, 0 = Damaged
    y_pred_test = np.where(raw_preds == -1, 0, 1)

    # 5. Metrics
    print("\n--- Isolation Forest Performance (Targeting Class 0) ---")
    print(classification_report(y_test_all, y_pred_test, target_names=['Damaged (0)', 'Healthy (1)']))
    
    # Get Anomaly Scores (Lower = More Anomalous/Damaged)
    # This is useful for your 3D plots later
    anomaly_scores = model.decision_function(X_test_all)
    
    # Save for the test loop
    # In your test cases, use: 
    # preds = (model.predict(X_test_wide) == -1).astype(int) 
    # (Note: make sure to map -1 to 0 and 1 to 1)

## Tune ROC Curve

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import auc, f1_score, roc_curve, precision_recall_curve

# 1. Determine Model Type and Get Probabilities for Class 1 (Healthy)
if hasattr(model, "predict_proba"):
    # XGBoost / Scikit-Learn path
    X_eval = X_test
    y_true = y_test
    y_pred_prob_healthy = model.predict_proba(X_eval)[:, 1]
else:
    # Neural Network (Keras) path
    # Note: Ensure batch_size is defined or use a default
    X_eval = X_te
    y_true = y_te
    y_pred_prob_healthy = model.predict(X_eval, batch_size=locals().get('batch_size', 32)).ravel()

# 2. Transform to Class 0 (Damaged) as our "Positive" target
# Since the model predicts Healthy=1, P(Damaged) = 1 - P(Healthy)
y_pred_prob_damaged = 1.0 - y_pred_prob_healthy

# --- ROC CURVE ---
# We specify pos_label=0 because we want to treat 'Damaged' as the detection target
fpr, tpr, thresholds_roc = roc_curve(y_true, y_pred_prob_damaged, pos_label=0)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 8))

plt.subplot(2, 1, 1)
plt.plot(fpr, tpr, label=f"ROC AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], "k--", linewidth=1)
plt.xlabel("False Positive Rate (Healthy as Defective)")
plt.ylabel("True Positive Rate (Correct Defective)")
plt.title("ROC Curve (Target: Defective)")
plt.legend(loc="lower right")

# --- PRECISION-RECALL CURVE ---
precision, recall, thresholds_pr = precision_recall_curve(y_true, y_pred_prob_damaged, pos_label=0)

plt.subplot(2, 1, 2)
plt.plot(recall, precision, label="Precision-Recall curve")
plt.xlabel("Recall (Detection Rate)")
plt.ylabel("Precision (Accuracy of Alarms)")
plt.title("P-R Curve (Target: Defective)")
plt.legend(loc="lower left")
plt.tight_layout()
plt.savefig("visualization/roc_auc_pr_curves_damaged_class.svg", dpi=300)
plt.show()

# --- THRESHOLD OPTIMIZATION ---
# We want the threshold that maximizes F1 for class 0
threshold_range = np.linspace(0.0, 1.0, 501)
f1_scores = []

for t in threshold_range:
    # Predict 0 (Damaged) if probability of 0 >= t
    y_pred_temp = (y_pred_prob_damaged >= t).astype(int)
    # Map back: if it's not Damaged(0), it's Healthy(1)
    # Actually, it's easier to just calculate F1 using pos_label=0
    # y_pred_temp is 1 if score >= t (meaning predicted damaged), else 0.
    # But y_true is 0 for damaged. So we swap y_pred_temp to match y_true's encoding:
    # If y_pred_temp=1 (we think it's damaged), the label should be 0.
    y_pred_labels = np.where(y_pred_temp == 1, 0, 1)
    
    f1 = f1_score(y_true, y_pred_labels, pos_label=0, zero_division=0)
    f1_scores.append(f1)

best_idx = np.argmax(f1_scores)
best_threshold = threshold_range[best_idx]
best_f1 = f1_scores[best_idx]

print(f"Optimization Results for 'Damaged' Class (0):")
print(f"Best F1-Score: {best_f1:.4f}")
print(f"Best Probability Threshold for Damaged: {best_threshold:.3f}")
print(f"Interpretation: If P(Damaged) > {best_threshold:.3f}, classify as Damaged.")

## Scikit NN

In [ ]:
# model = sklearn.neural_network.MLPClassifier(
#     hidden_layer_sizes=(1000,500),
#     verbose=1,
# )
# model.fit(X_train, y_train)

## Detailed report

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

# Predict on test set
if hasattr(model, "predict_proba"):
    X_eval = X_test
    y_pred_prob = model.predict_proba(X_eval)[:, 1]
else:
    X_eval = X_te
    y_pred_prob = model.predict(X_eval, batch_size=batch_size).ravel()

DECISION_THRESHOLD = 0.975
y_pred = (y_pred_prob >= DECISION_THRESHOLD).astype(int)

# Print detailed classification report
print("Classification Report:")
report = classification_report(y_test, y_pred, output_dict=True)
print(classification_report(y_test, y_pred))

# Print confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

# --- Plot Confusion Matrix ---
plt.figure(figsize=(4, 3))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, 
            xticklabels=["Pred Defective", "Pred Perfect"], yticklabels=["True Defective", "True Perfect"])
plt.title("Confusion Matrix")
plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.tight_layout()
plt.savefig("visualization/confusion_matrix.svg", dpi=300)
plt.show()

# --- Plot F1-score, Precision, Recall ---
metrics = ["precision", "recall", "f1-score"]
class_names = {0: "Defective", 1: "Perfect"}
classes = [str(c) for c in sorted(np.unique(y_test))]
scores = {m: [report[c][m] for c in classes] for m in metrics}

# Map numeric classes to readable labels for the x-axis
display_labels = [class_names.get(int(float(c)), c) for c in classes]

plt.figure(figsize=(6, 4))
x = np.arange(len(classes))
width = 0.2
for i, m in enumerate(metrics):
    plt.bar(x + i*width, scores[m], width=width, label=m.capitalize())
plt.xticks(x + width, display_labels)
plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.xlabel("Class")
plt.title("Precision, Recall, F1-score per Class")
plt.legend(loc="upper center")
plt.tight_layout()
plt.savefig("visualization/classification_metrics.svg", dpi=300)
plt.show()

n = len(y_test)
y_test = y_test.reset_index(drop=True)
y_pred = np.asarray(y_pred)

# Apply small vertical offsets so markers/lines don't overlap when plotted
y_test_show = pd.Series(y_test.astype(float).values - 0.03, name=y_test.name)
y_pred_show = (np.asarray(y_pred).astype(float) + 0.03)

# Make markers and lines more visible for the following plot
plt.rcParams.update({"lines.markersize": 6, "lines.linewidth": 1.5})

# Show how many mismatches are in the displayed subset
mismatches = (np.round(y_test_show + 0.03).astype(int) != np.round(y_pred_show - 0.03).astype(int)).sum()
print(f"Showing {len(y_test_show)} samples (subset). Mismatches in view: {mismatches}")
# --- Plot individual predictions vs actual values ---
plt.figure(figsize=(12, 3))
plt.plot(y_test_show.values, 'o-', label="Actual", alpha=0.7)
plt.plot(y_pred_show, 'x-', label="Predicted", alpha=0.7)
plt.title("Individual Predictions vs Actual Values")
plt.xlabel("Sample Index")
plt.ylabel("Class")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
from constants import NODES_TO_BEAMS
import numpy as np
import pandas as pd

def nodes_to_beam_labels(node_numbers, node_labels):
    """
    Convert node-level labels to beam-level labels.
    Rule: beam is defective (0) if ANY associated node is defective.
    """
    beam_to_labels = {}

    for node, lbl in zip(node_numbers, node_labels):
        if node not in NODES_TO_BEAMS:
            continue
        beams = np.atleast_1d(NODES_TO_BEAMS[node])
        for b in beams:
            beam_to_labels.setdefault(b, []).append(lbl)

    beam_ids = sorted(beam_to_labels.keys())
    beam_labels = [
        0 if np.any(np.array(beam_to_labels[b]) == 0) else 1
        for b in beam_ids
    ]

    return np.array(beam_ids), np.array(beam_labels)
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# -----------------------------
# Predict node-level probabilities
# -----------------------------
if hasattr(model, "predict_proba"):
    y_pred_prob = model.predict_proba(X_test)[:, 1]
else:
    y_pred_prob = model.predict(X_test, batch_size=batch_size).ravel()

DECISION_THRESHOLD = 0.975
y_pred_node = (y_pred_prob >= DECISION_THRESHOLD).astype(int)

# -----------------------------
# Convert to beam-level labels
# -----------------------------
beam_ids, y_test_beam = nodes_to_beam_labels(
    node_numbers_test, y_test.values
)
_, y_pred_beam = nodes_to_beam_labels(
    node_numbers_test, y_pred_node
)

print(f"Evaluating {len(beam_ids)} beams")

# -----------------------------
# Beam-wise classification report
# -----------------------------
print("\nBeam-level Classification Report:")
print(classification_report(
    y_test_beam, y_pred_beam,
    target_names=["Defective", "Perfect"]
))

report = classification_report(
    y_test_beam, y_pred_beam,
    target_names=["Defective", "Perfect"],
    output_dict=True
)

# -----------------------------
# Beam-wise confusion matrix
# -----------------------------
cm = confusion_matrix(y_test_beam, y_pred_beam)

plt.figure(figsize=(4, 3))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues", cbar=False,
    xticklabels=["Pred Defective", "Pred Perfect"],
    yticklabels=["True Defective", "True Perfect"],
)
plt.title("Beam-level Confusion Matrix")
plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.tight_layout()
plt.savefig("visualization/confusion_matrix_beam.svg", dpi=1000)
plt.show()

# -----------------------------
# Beam-wise metrics bar plot
# -----------------------------
metrics = ["precision", "recall", "f1-score"]
classes = ["Defective", "Perfect"]
scores = {m: [report[c][m] for c in classes] for m in metrics}

plt.figure(figsize=(6, 4))
x = np.arange(len(classes))
width = 0.2

for i, m in enumerate(metrics):
    plt.bar(x + i * width, scores[m], width=width, label=m.capitalize())

plt.xticks(x + width, classes)
plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.xlabel("Beam Class")
plt.title("Beam-level Precision, Recall, F1-score")
plt.legend(loc="upper center")
plt.tight_layout()
plt.savefig("visualization/classification_metrics_beam.svg", dpi=1000)
plt.show()


In [ ]:
from constants import DEFECTIVE_NODES_BY_REGION, NODES_TO_BEAMS, BEAMS_TO_NODES
from utils import scenario_number_to_string

# Visualize true vs predicted labels on bridge nodes (test set)#
for scenario in scenarios_test:
    region = scenario_number_to_combination(scenario)[-1]
    defective_nodes = DEFECTIVE_NODES_BY_REGION[region]

    y_test_sc = y_test_by_scenario[scenario]
    x_test_sc = X_test_by_scenario[scenario].drop(columns=[y_var,"scenario"])
    node_numbers_sc = x_test_sc["Node Number"].values
    if DROP_NODE_NUMBERS:
        x_test_sc = x_test_sc.drop(columns=["Node Number"])
    y_pred_sc_prob = model.predict_proba(x_test_sc)[:, 1]
    y_pred_sc = (y_pred_sc_prob >= DECISION_THRESHOLD).astype(int)
    highlight_nodes_pred = node_numbers_sc[y_pred_sc == 0]

    # Aggregate by beams
    print([NODES_TO_BEAMS[node] for node in highlight_nodes_pred if node in NODES_TO_BEAMS])
    highlight_beam_lists = [np.atleast_1d(NODES_TO_BEAMS[node]) for node in highlight_nodes_pred if node in NODES_TO_BEAMS]
    true_beam_lists = [np.atleast_1d(NODES_TO_BEAMS[node]) for node in defective_nodes if node in NODES_TO_BEAMS]

    highlight_beams_pred = np.concatenate(highlight_beam_lists) if highlight_beam_lists else np.array([], dtype=int)
    true_beams = np.concatenate(true_beam_lists) if true_beam_lists else np.array([], dtype=int)
    # Calculate error metrics at beam level
    true_positives = len(set(highlight_beams_pred) & set(true_beams))
    false_positives = len(set(highlight_beams_pred) - set(true_beams))
    false_negatives = len(set(true_beams) - set(highlight_beams_pred))
    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0.0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    print(f"Scenario {scenario} (Region {region}): Beam-level Precision: {precision:.4f}, Recall: {recall:.4f}, F1-score: {f1:.4f}")

    highlight_nodes_by_beams = np.concatenate(
        [np.atleast_1d(BEAMS_TO_NODES[beam]) for beam in highlight_beams_pred if beam in BEAMS_TO_NODES]) if highlight_beams_pred.size > 0 else np.array([], dtype=int
    )

    plot_bridge_3d_structure(
        highlight_nodes=defective_nodes, highlight_color="red",
        title=f"True Defective Nodes - {scenario_number_to_string(scenario)})"
    )
    plot_bridge_3d_structure(
        highlight_nodes=highlight_nodes_pred, highlight_color="lime",
        title=f"Predicted Defective Nodes - {scenario_number_to_string(scenario)}"
    )

# Evaluate on test-cases

In [ ]:
for test_case in TEST_CASES:
    df_test = read_test_case(test_case)
    
    # 1. Select initial columns
    columns = ["Node Number", "time", "X", "Y", "Z", "DirectionalDeformation_Z_axis"] 
    df_test = df_test[columns]

    # --- START LOCAL INFORMATION INTEGRATION ---
    if ADD_LOCAL_INFORMATION:
        # Create Pivot for vectorized neighbor mean calculation
        df_wide_temp = df_test.pivot_table(
            index=['time'], 
            columns='Node Number', 
            values='DirectionalDeformation_Z_axis'
        )

        local_mean_wide = pd.DataFrame(index=df_wide_temp.index, columns=df_wide_temp.columns)
        local_std_wide = pd.DataFrame(index=df_wide_temp.index, columns=df_wide_temp.columns)

        for node_id in df_wide_temp.columns:
            if node_id in neighbor_map:
                neighbors = [n for n in neighbor_map[node_id] if n in df_wide_temp.columns]
                if neighbors:
                    local_mean_wide[node_id] = df_wide_temp[neighbors].mean(axis=1)
                    local_std_wide[node_id] = df_wide_temp[neighbors].std(axis=1)

        local_mean_long = local_mean_wide.reset_index().melt(
            id_vars=['time'],
            var_name='Node Number',
            value_name='Z_local_mean'
        )
        local_std_long = local_std_wide.reset_index().melt(
            id_vars=['time'],
            var_name='Node Number',
            value_name='Z_local_std'
        )
        
        local_mean_long['Node Number'] = local_mean_long['Node Number'].astype(df_test['Node Number'].dtype)
        local_std_long['Node Number'] = local_std_long['Node Number'].astype(df_test['Node Number'].dtype)
        df_test = pd.merge(df_test, local_mean_long, on=['time', 'Node Number'], how='left')
        df_test = pd.merge(df_test, local_std_long, on=['time', 'Node Number'], how='left')
        
        # Calculate the residual feature
        df_test['Z_neighbor_residual'] = df_test['DirectionalDeformation_Z_axis'] - df_test['Z_local_mean']
    # --- END LOCAL INFORMATION INTEGRATION ---

    # 2. Standardization
    # Ensure variable_data contains EXACTLY the same columns used during scaler.fit()
    # Usually: ['X', 'Y', 'Z', 'DirectionalDeformation_Z_axis', 'Z_local_mean', 'Z_neighbor_residual']
    variable_data = df_test.drop(columns=["Node Number", "time"])
    df_test_std_values = scaler.transform(variable_data)
    df_test_std = pd.DataFrame(
        df_test_std_values, 
        columns=variable_data.columns, 
        index=variable_data.index
    )
    if DROP_COORDINATES:
        df_test_std = df_test_std.drop(columns=["X", "Y", "Z"])
    
    # 3. Re-add Metadata
    df_test_std["scenario"] = -1
    for col in ["Node Number", "time"]:
        df_test_std[col] = df_test[col]
    
    del df_test # save RAM

    # 4. Reshape to Wide (Time-series features)
    X_test_wide = reshape_multi_variable_to_wide_nodes(
        df_test_std, y_var_name=y_var, use_coords=not DROP_COORDINATES
    )
    
    # Preserve node numbers for visualization before dropping from X
    node_numbers = X_test_wide["Node Number"].values 
    
    # Prepare X for XGBoost
    cols_to_drop = ["scenario", "delta_health"]
    if DROP_NODE_NUMBERS:
        cols_to_drop.append("Node Number")
    
    X_test_wide.drop(columns=[c for c in cols_to_drop if c in X_test_wide.columns], inplace=True)

    # 5. Predict
    if CLASSIFIER_TO_USE not in ("NN", "AE"):
        y_pred_test_prob = model.predict_proba(X_test_wide)[:, 1]
    else:
        y_pred_test_prob = model.predict(X_test_wide, batch_size=batch_size).ravel()
    y_pred_test = (y_pred_test_prob >= DECISION_THRESHOLD).astype(int)
    
    # 0 is damaged, 1 is healthy based on your previous logic
    damaged_indices = np.where(y_pred_test == 0)[0]
    print(f"Predicted damaged nodes for test case {test_case}: {len(damaged_indices)}")

    # 6. Plot
    highlight_nodes_pred = node_numbers[damaged_indices]
    # Convert nodes to beams
    highlight_beam_lists = [np.atleast_1d(NODES_TO_BEAMS[node]) for node in highlight_nodes_pred if node in NODES_TO_BEAMS]
    highlight_beams_pred = np.concatenate(highlight_beam_lists) if highlight_beam_lists else np.array([], dtype=int)
    highlight_nodes_by_beams = np.concatenate(
        [np.atleast_1d(BEAMS_TO_NODES[beam]) for beam in highlight_beams_pred if beam in BEAMS_TO_NODES]) if highlight_beams_pred.size > 0 else np.array([], dtype=int
    )
    plot_bridge_3d_structure(
        highlight_nodes=highlight_nodes_by_beams,
        highlight_color="lime",
        annotations=NODES_TO_BEAMS,
        title=f"Predicted Damaged Nodes for Test Case {test_case}"
    )
    # Plot the prediction probabilities as a color map
    plot_bridge_3d_structure(
        highlight_nodes=highlight_nodes_by_beams,
        highlight_color="lime",
        annotations=NODES_TO_BEAMS,
        color_scale={node_number: prob for node_number, prob in zip(node_numbers, y_pred_test_prob)},
        title=f"Predicted Damaged Beams for Test Case {test_case}",
        s=2
    )

In [ ]:
# plot_bridge_3d_structure(
#     highlight_nodes=np.concatenate(list(DEFECTIVE_NODES_BY_REGION.values())),
#     highlight_color="lime",
#     annotations=NODES_TO_BEAMS
# )